# Mamba-2 State Injection 가능 여부 검증

## 핵심 질문
**"문서를 읽혀서 만든 state를 저장했다가 나중에 주입하면, 모델이 그 문서를 읽은 것처럼 동작하는가?"**

## 4가지 Setup 비교
```
A (Oracle)   : [문서 전체 + query] → 한 번에 forward → 생성
B (Proposed) : [문서 → state 저장] → [state 주입 + query] → 생성  ← 검증 대상
C (No context): [query만] → 생성  ← 하한선
D (Wrong state): [무관한 문서 state 주입 + query] → 생성  ← sanity check
```

## 판단 기준
```
A ≈ B >> C → state injection 작동 → RAG 구현 진행
A >> B ≈ C → state injection 안 됨 → 방향 재고
B ≈ D      → state가 query 생성에 영향 자체를 안 줌
```

## 1. Import & 모델 로드

In [1]:
import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer
from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel
from mamba_ssm.utils.generation import InferenceParams
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

ModuleNotFoundError: No module named 'transformers'

In [ ]:
MODEL_NAME = "state-spaces/mamba2-130m"

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")
tokenizer.pad_token = tokenizer.eos_token

model = MambaLMHeadModel.from_pretrained(MODEL_NAME, device=device, dtype=torch.float32)
model.eval()

mixer    = model.backbone.layers[0].mixer
cfg      = model.config
N_LAYERS = cfg.n_layer
D_STATE  = mixer.d_state
HEADDIM  = mixer.headdim
N_HEADS  = mixer.nheads

print(f"n_layers={N_LAYERS}, n_heads={N_HEADS}, headdim={HEADDIM}, d_state={D_STATE}")

## 2. 실험 데이터

문서 + 해당 문서에 대한 질문/답을 직접 구성.
답을 알려면 반드시 문서를 읽어야 하는 질문들.

In [2]:
QA_PAIRS = [
    {
        "doc": """The Apollo 11 mission was the first crewed lunar landing mission. 
It launched on July 16, 1969, and landed on the Moon on July 20, 1969. 
The crew consisted of Neil Armstrong, Buzz Aldrin, and Michael Collins. 
Neil Armstrong was the first human to walk on the Moon. 
The landing site was called Tranquility Base in the Sea of Tranquility.""",
        "query": "Where did Apollo 11 land on the Moon?",
        "answer_keywords": ["tranquility", "sea of tranquility"],
    },
    {
        "doc": """The Python programming language was created by Guido van Rossum 
and first released in 1991. Python emphasizes code readability and uses 
significant indentation. It supports multiple programming paradigms including 
procedural, object-oriented, and functional programming. Python is widely used 
in data science, web development, and artificial intelligence.""",
        "query": "Who created the Python programming language?",
        "answer_keywords": ["guido", "van rossum"],
    },
    {
        "doc": """The Great Wall of China was built over many centuries by various 
Chinese dynasties. Construction began as early as the 7th century BC. 
The most well-known sections were built during the Ming Dynasty between 
1368 and 1644. The wall stretches approximately 21,196 kilometers in total length. 
It was primarily built to protect against nomadic invasions from the north.""",
        "query": "Which dynasty built the most well-known sections of the Great Wall?",
        "answer_keywords": ["ming"],
    },
    {
        "doc": """Marie Curie was a Polish-French physicist and chemist who conducted 
pioneering research on radioactivity. She was the first woman to win a Nobel Prize, 
and the only person to win Nobel Prizes in two different sciences. She won the 
Nobel Prize in Physics in 1903 and the Nobel Prize in Chemistry in 1911. 
She discovered the elements polonium and radium.""",
        "query": "What two elements did Marie Curie discover?",
        "answer_keywords": ["polonium", "radium"],
    },
    {
        "doc": """The Amazon River in South America is the largest river in the world 
by discharge volume of water. It originates in the Andes mountains of Peru 
and flows eastward through Brazil before emptying into the Atlantic Ocean. 
The river basin covers approximately 7 million square kilometers. 
The Amazon rainforest surrounding it contains about 10 percent of all species on Earth.""",
        "query": "Where does the Amazon River originate?",
        "answer_keywords": ["andes", "peru"],
    },
]

# 무관한 문서 (Setup D용 noise state)
IRRELEVANT_DOC = """The history of chess dates back nearly 1500 years. 
The game originated in India before spreading to Persia, the Arab world, 
and eventually to Europe. Chess is played on an 8x8 board with 16 pieces 
per player. The objective is to checkmate the opponent's king."""

print(f"QA pairs: {len(QA_PAIRS)}")

QA pairs: 5


## 3. 핵심 함수

### State 추출 / 주입 방법
```
InferenceParams.key_value_memory_dict 를 활용:
  - 문서 forward → state 추출 후 저장
  - query forward 시 저장된 state를 initial state로 세팅
```

In [3]:
def extract_state(model, text, device, max_len=256):
    """문서를 읽혀서 SSM state 추출."""
    ids = tokenizer.encode(text, return_tensors='pt')[:, :max_len].to(device)
    inf = InferenceParams(max_seqlen=ids.shape[1], max_batch_size=1)
    with torch.no_grad():
        _ = model(ids, inference_params=inf)
    # {layer_idx: (conv_state, ssm_state)}
    # ssm_state shape: (1, nheads, headdim, d_state)
    state = {
        k: (v[0].clone(), v[1].clone())
        for k, v in inf.key_value_memory_dict.items()
    }
    return state


def inject_state_and_generate(model, query_text, injected_state, device,
                               max_new_tokens=50, doc_token_len=256):
    """
    핵심 수정:
    1. dummy forward로 key_value_memory_dict 구조 초기화
    2. 초기화된 dict에 저장된 state 덮어쓰기
    3. seqlen_offset = doc_token_len 으로 설정
       → 모델이 이미 doc_token_len개 토큰을 처리한 것으로 인식
    4. query를 token by token으로 처리 (step mode)
    """
    query_ids = tokenizer.encode(query_text, return_tensors='pt').to(device)
    q_len     = query_ids.shape[1]

    inf = InferenceParams(
        max_seqlen   = doc_token_len + q_len + max_new_tokens,
        max_batch_size = 1
    )

    # Step 1: dummy single token forward → dict 구조 초기화
    dummy = torch.zeros(1, 1, dtype=torch.long).to(device)
    with torch.no_grad():
        _ = model(dummy, inference_params=inf)

    # Step 2: 저장된 state로 덮어쓰기
    if injected_state is not None:
        for layer_idx, (conv_s, ssm_s) in injected_state.items():
            if layer_idx in inf.key_value_memory_dict:
                inf.key_value_memory_dict[layer_idx] = (
                    conv_s.clone().to(device),
                    ssm_s.clone().to(device)
                )

    # Step 3: seqlen_offset = doc 길이로 설정
    # → 모델이 "이미 doc를 읽은 상태"로 인식
    inf.seqlen_offset = doc_token_len

    # Step 4: query를 token by token으로 처리
    for i in range(q_len):
        token = query_ids[:, i:i+1]
        with torch.no_grad():
            _ = model(token, inference_params=inf)
        inf.seqlen_offset += 1

    # Step 5: generation
    generated  = []
    next_token = query_ids[:, -1:]

    for _ in range(max_new_tokens):
        with torch.no_grad():
            out = model(next_token, inference_params=inf)
        logits     = out[0][:, -1, :]
        next_token = logits.argmax(dim=-1, keepdim=True)
        generated.append(next_token.item())
        inf.seqlen_offset += 1
        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def run_setup_A(model, doc, query, device, doc_max_len=256, max_new_tokens=50):
    """Setup A (Oracle): [문서 + query] 한 번에 forward → 생성."""
    combined_text = doc + "\n\nQuestion: " + query + "\nAnswer:"
    combined_ids  = tokenizer.encode(combined_text, return_tensors='pt').to(device)

    inf = InferenceParams(
        max_seqlen=combined_ids.shape[1] + max_new_tokens,
        max_batch_size=1
    )

    with torch.no_grad():
        _ = model(combined_ids, inference_params=inf)
    inf.seqlen_offset = combined_ids.shape[1]

    generated  = combined_ids.clone()
    next_token = combined_ids[:, -1:]

    for _ in range(max_new_tokens):
        with torch.no_grad():
            out = model(next_token, inference_params=inf)
        logits     = out[0][:, -1, :]
        next_token = logits.argmax(dim=-1, keepdim=True)
        generated  = torch.cat([generated, next_token], dim=1)
        inf.seqlen_offset += 1
        if next_token.item() == tokenizer.eos_token_id:
            break

    gen_ids  = generated[:, combined_ids.shape[1]:]
    return tokenizer.decode(gen_ids[0], skip_special_tokens=True).strip()


def check_answer(generated_text, keywords):
    """생성된 텍스트에 정답 키워드가 포함됐는지 체크."""
    text_lower = generated_text.lower()
    return any(kw.lower() in text_lower for kw in keywords)

## 4. Sanity Check: State가 실제로 주입되는가

In [10]:
# Sanity check 재실험
query_ids = tokenizer.encode(test_query, return_tensors='pt').to(device)

# no injection
inf_no = InferenceParams(max_seqlen=500, max_batch_size=1)
dummy  = torch.zeros(1, 1, dtype=torch.long).to(device)
with torch.no_grad():
    _ = model(dummy, inference_params=inf_no)
state_no = inf_no.key_value_memory_dict[0][1].clone().cpu()

# with injection
inf_yes = InferenceParams(max_seqlen=500, max_batch_size=1)
with torch.no_grad():
    _ = model(dummy, inference_params=inf_yes)
# 덮어쓰기
for layer_idx, (conv_s, ssm_s) in doc_state.items():
    if layer_idx in inf_yes.key_value_memory_dict:
        inf_yes.key_value_memory_dict[layer_idx] = (
            conv_s.clone().to(device),
            ssm_s.clone().to(device)
        )
state_yes = inf_yes.key_value_memory_dict[0][1].clone().cpu()

diff = (state_yes - state_no).abs().mean().item()
print(f"State diff: {diff:.6f}")
print(f"→ {'✅ 주입 성공' if diff > 1e-5 else '❌ 여전히 실패'}")

State diff: 0.051281
→ ✅ 주입 성공


## 5. 메인 실험: 4가지 Setup 비교

In [16]:
MODEL_LIST = [
    "state-spaces/mamba2-130m",
    "state-spaces/mamba2-370m",
    "state-spaces/mamba2-780m",
    "state-spaces/mamba2-1.3b",
    # "state-spaces/mamba2-2.7b",  
]

all_model_results = {}

for MODEL_NAME in MODEL_LIST:
    print(f"\n{'='*60}")
    print(f"Model: {MODEL_NAME}")
    print(f"{'='*60}")

    # 모델 로드
    model = MambaLMHeadModel.from_pretrained(
        MODEL_NAME, device=device, dtype=torch.float32
    )
    model.eval()

    irrel_state, irrel_n = extract_state(
        model, IRRELEVANT_DOC, device, max_len=DOC_MAX_LEN
    )

    results = []
    hits = {'A': 0, 'B': 0, 'C': 0, 'D': 0}

    for qi, qa in enumerate(QA_PAIRS):
        doc_state, doc_n = extract_state(model, qa['doc'], device, max_len=DOC_MAX_LEN)

        gen_A = run_setup_A(model, qa['doc'], qa['query'], device, DOC_MAX_LEN, MAX_NEW_TOKENS)
        gen_B = run_setup_B(model, doc_state, doc_n, qa['query'], device, MAX_NEW_TOKENS)
        gen_C = run_setup_C(model, qa['query'], device, MAX_NEW_TOKENS)
        gen_D = run_setup_D(model, irrel_state, irrel_n, qa['query'], device, MAX_NEW_TOKENS)

        row = {}
        for s, gen in zip(['A','B','C','D'], [gen_A, gen_B, gen_C, gen_D]):
            hit = check_answer(gen, qa['answer_keywords'])
            row[s] = {'gen': gen, 'hit': hit}
            hits[s] += hit

        results.append(row)
        print(f"  Q{qi+1} A={'✓' if row['A']['hit'] else '✗'} "
              f"B={'✓' if row['B']['hit'] else '✗'} "
              f"C={'✓' if row['C']['hit'] else '✗'} "
              f"D={'✓' if row['D']['hit'] else '✗'}")

    tag = MODEL_NAME.split('/')[-1]
    all_model_results[tag] = {
        'hits': hits,
        'acc': {s: hits[s]/len(QA_PAIRS) for s in 'ABCD'},
        'results': results,
    }

    # 모델 unload
    del model
    torch.cuda.empty_cache()

# ── 최종 비교 테이블 ──────────────────────────────────────────────
print(f"\n{'='*60}")
print("SCALING RESULTS")
print(f"{'='*60}")
print(f"{'Model':<18} | {'A Oracle':>9} | {'B Inject':>9} | {'C None':>7} | {'D Wrong':>8} | {'B-C gap':>8}")
print("-" * 70)

for tag, res in all_model_results.items():
    acc  = res['acc']
    gap  = acc['B'] - acc['C']
    flag = '✅' if gap > 0.15 else ('⚠️' if gap > 0 else '❌')
    print(f"{tag:<18} | {acc['A']:>9.0%} | {acc['B']:>9.0%} | "
          f"{acc['C']:>7.0%} | {acc['D']:>8.0%} | {gap:>+7.0%} {flag}")


Model: state-spaces/mamba2-130m


ValueError: too many values to unpack (expected 2)

In [11]:
# 무관한 문서의 state 미리 추출 (Setup D용)
irrelevant_state = extract_state(model, IRRELEVANT_DOC, device)

MAX_NEW_TOKENS = 40
DOC_MAX_LEN    = 256

results = []

print(f"{'='*70}")
print(f"{'QA #':<5} | {'Setup':<12} | {'Hit':^5} | {'Generated text'}")
print(f"{'='*70}")

for qa_idx, qa in enumerate(QA_PAIRS):
    doc     = qa['doc']
    query   = "Question: " + qa['query'] + "\nAnswer:"
    keywords = qa['answer_keywords']

    # 문서 state 추출
    doc_state = extract_state(model, doc, device, max_len=DOC_MAX_LEN)

    row = {'qa_idx': qa_idx, 'question': qa['query'], 'keywords': keywords}

    # ── Setup A: Oracle ──────────────────────────────────────────────
    gen_A = run_setup_A(model, doc, qa['query'], device,
                        doc_max_len=DOC_MAX_LEN, max_new_tokens=MAX_NEW_TOKENS)
    hit_A = check_answer(gen_A, keywords)
    row['A'] = {'gen': gen_A, 'hit': hit_A}
    print(f"QA {qa_idx+1:<3} | {'A (Oracle)':<12} | {'✓' if hit_A else '✗':^5} | {gen_A[:60]}")

    # ── Setup B: State injection ──────────────────────────────────────
    gen_B = inject_state_and_generate(model, query, doc_state, device,
                                       max_new_tokens=MAX_NEW_TOKENS)
    hit_B = check_answer(gen_B, keywords)
    row['B'] = {'gen': gen_B, 'hit': hit_B}
    print(f"{'':5} | {'B (Inject)':<12} | {'✓' if hit_B else '✗':^5} | {gen_B[:60]}")

    # ── Setup C: No context ───────────────────────────────────────────
    gen_C = inject_state_and_generate(model, query, None, device,
                                       max_new_tokens=MAX_NEW_TOKENS)
    hit_C = check_answer(gen_C, keywords)
    row['C'] = {'gen': gen_C, 'hit': hit_C}
    print(f"{'':5} | {'C (No ctx)':<12} | {'✓' if hit_C else '✗':^5} | {gen_C[:60]}")

    # ── Setup D: Wrong state ──────────────────────────────────────────
    gen_D = inject_state_and_generate(model, query, irrelevant_state, device,
                                       max_new_tokens=MAX_NEW_TOKENS)
    hit_D = check_answer(gen_D, keywords)
    row['D'] = {'gen': gen_D, 'hit': hit_D}
    print(f"{'':5} | {'D (Wrong)':<12} | {'✓' if hit_D else '✗':^5} | {gen_D[:60]}")
    print(f"{'-'*70}")

    results.append(row)

QA #  | Setup        |  Hit  | Generated text
QA 1   | A (Oracle)   |   ✓   | The landing site was called Tranquility Base in the Sea of T
      | B (Inject)   |   ✓   | The landing site was called Tranquility Base in the Sea of T
      | C (No ctx)   |   ✗   | Apollo 11 was the first manned mission to the Moon.
Question
      | D (Wrong)    |   ✗   | Apollo 11 was a space shuttle mission. Apollo 11 was the fir
----------------------------------------------------------------------
QA 2   | A (Oracle)   |   ✓   | The Python programming language was created by Guido van Ros
      | B (Inject)   |   ✓   | The Python programming language was created by Guido van Ros
      | C (No ctx)   |   ✗   | A:

The Python programming language is a programming languag
      | D (Wrong)    |   ✗   | The Python programming language was created by the famous pr
----------------------------------------------------------------------
QA 3   | A (Oracle)   |   ✗   | The Great Wall of China was built over man

## 6. 결과 요약

In [13]:
setups = ['A', 'B', 'C', 'D']
setup_labels = {
    'A': 'Oracle (doc+query)',
    'B': 'State Injection',
    'C': 'No Context',
    'D': 'Wrong State',
}

hit_counts = {s: sum(r[s]['hit'] for r in results) for s in setups}
total      = len(results)

print("=" * 50)
print("SUMMARY")
print("=" * 50)
for s in setups:
    acc = hit_counts[s] / total
    bar = '█' * hit_counts[s] + '░' * (total - hit_counts[s])
    print(f"{setup_labels[s]:<25}: {hit_counts[s]}/{total} [{bar}] {acc:.0%}")

print()
print("판단:")
A_acc = hit_counts['A'] / total
B_acc = hit_counts['B'] / total
C_acc = hit_counts['C'] / total
D_acc = hit_counts['D'] / total

if B_acc >= A_acc * 0.7 and B_acc > C_acc + 0.1:
    print("  ✅ State injection 유효 → RAG 구현 진행")
elif B_acc > C_acc + 0.05:
    print("  ⚠️  State injection 약하게 유효 → 추가 실험 필요")
elif abs(B_acc - D_acc) < 0.1:
    print("  ❌ B ≈ D: state가 생성에 영향을 안 줌 → injection 메커니즘 문제")
else:
    print("  ❌ B ≈ C: state injection 효과 없음 → 방향 재고 필요")

SUMMARY
Oracle (doc+query)       : 4/5 [████░] 80%
State Injection          : 4/5 [████░] 80%
No Context               : 0/5 [░░░░░] 0%
Wrong State              : 0/5 [░░░░░] 0%

판단:
  ✅ State injection 유효 → RAG 구현 진행


## 7. 상세 출력: 어느 QA에서 B가 성공/실패했는가

In [14]:
print("=" * 70)
print("DETAILED RESULTS")
print("=" * 70)

for r in results:
    print(f"\nQ{r['qa_idx']+1}: {r['question']}")
    print(f"     Keywords: {r['keywords']}")
    for s in setups:
        hit_mark = '✓' if r[s]['hit'] else '✗'
        print(f"     [{s}] {hit_mark} {r[s]['gen'][:80]}")

DETAILED RESULTS

Q1: Where did Apollo 11 land on the Moon?
     Keywords: ['tranquility', 'sea of tranquility']
     [A] ✓ The landing site was called Tranquility Base in the Sea of Tranquility.

Questio
     [B] ✓ The landing site was called Tranquility Base in the Sea of Tranquility.Question:
     [C] ✗ Apollo 11 was the first manned mission to the Moon.
Question: Where did Apollo 1
     [D] ✗ Apollo 11 was a space shuttle mission. Apollo 11 was the first manned space flig

Q2: Who created the Python programming language?
     Keywords: ['guido', 'van rossum']
     [A] ✓ The Python programming language was created by Guido van Rossum and first 
relea
     [B] ✓ The Python programming language was created by Guido van Rossum and first 
relea
     [C] ✗ A:

The Python programming language is a programming language that is used to cr
     [D] ✗ The Python programming language was created by the famous programmer, 
Steve Job

Q3: Which dynasty built the most well-known sections of the G

## 8. 추가 분석: State injection depth 효과

**"모든 layer에 주입 vs 특정 layer만 주입"**

Type A head가 많은 layer (6, 10, 18 등)만 주입하면 어떤가?

In [15]:
import torch.nn.functional as F

# slow head 많은 layer 찾기
def get_slow_layers(model, n_layers, slow_thr=0.99, min_heads=1):
    slow_layers = []
    for i in range(n_layers):
        m      = model.backbone.layers[i].mixer
        A_log  = m.A_log.detach().cpu()
        dt     = F.softplus(m.dt_bias.detach().cpu())
        A_disc = torch.exp(-torch.exp(A_log) * dt)
        n_slow = (A_disc >= slow_thr).sum().item()
        if n_slow >= min_heads:
            slow_layers.append(i)
    return slow_layers

slow_layers = get_slow_layers(model, N_LAYERS)
print(f"Slow layers: {slow_layers}")


def inject_selective_and_generate(model, query_text, full_state, layer_indices,
                                   device, max_new_tokens=40):
    """특정 layer index에만 state 주입."""
    # 선택된 layer만 포함하는 partial state
    partial_state = {li: full_state[li] for li in layer_indices if li in full_state}
    return inject_state_and_generate(
        model, query_text, partial_state, device, max_new_tokens
    )


# 첫 번째 QA로 layer subset 실험
qa   = QA_PAIRS[0]
doc  = qa['doc']
query = "Question: " + qa['query'] + "\nAnswer:"
kw   = qa['answer_keywords']

doc_state = extract_state(model, doc, device)

configs = {
    'All layers'       : list(range(N_LAYERS)),
    'Slow layers only' : slow_layers,
    'Early (0-7)'      : list(range(8)),
    'Late (16-23)'     : list(range(16, N_LAYERS)),
    'Layer 18 only'    : [18],  # semantic specialist head가 있던 layer
}

print(f"\nQ: {qa['query']}")
print(f"Keywords: {kw}\n")
print(f"{'Config':<22} | {'Hit':^5} | Generated text")
print("-" * 70)

for config_name, layer_list in configs.items():
    gen = inject_selective_and_generate(
        model, query, doc_state, layer_list, device
    )
    hit = check_answer(gen, kw)
    print(f"{config_name:<22} | {'✓' if hit else '✗':^5} | {gen[:55]}")

Slow layers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22]

Q: Where did Apollo 11 land on the Moon?
Keywords: ['tranquility', 'sea of tranquility']

Config                 |  Hit  | Generated text
----------------------------------------------------------------------
All layers             |   ✓   | The landing site was called Tranquility Base in the Sea
Slow layers only       |   ✓   | Apollo 11 landed on the Moon on July 16, 1969.

A:

The
Early (0-7)            |   ✗   | The Apollo 11 landing site was located on the Moon.
Que
Late (16-23)           |   ✗   | Apollo 11 landed on the Moon on July 4, 1969.
Question:
Layer 18 only          |   ✗   | The Apollo 11 landing site was located on the Moon.
Que
